# HumanAI ISSR4 — NSF FOA Exploration Notebook

## Purpose of this notebook
This notebook documents my exploration of **multiple tagging strategies for a single NSF funding opportunity URL**.  
The goal is not only to show a final output, but also to make the reasoning transparent for maintainers:

- how the NSF page is ingested,
- how structured FOA fields are extracted,
- how tags can be assigned using different methods,
- and which approach is most appropriate for the actual task constraints.

## Why this notebook is structured this way
Because the notebook became long during experimentation, I have organized it as a **progressive evaluation of approaches** rather than a raw sequence of trials.

The progression is intentional:

1. **Deterministic parsing + rule-based tagging**  
   This is the closest to the task requirement and easiest to validate.
2. **TF-IDF similarity over ontology labels**  
   This introduces a lightweight statistical baseline.
3. **Embedding-based semantic matching**  
   This captures wording variation better than exact keyword matching.
4. **Hybrid variants**  
   These show how deterministic outputs can be preserved while still benefiting from semantic recovery.

## Maintainer reading guide
If you are reviewing quickly, the most useful order is:

- read the **comparison table** below,
- inspect **Approach 1** first for the task-aligned baseline,
- then skim **Approach 2** and **Approach 3** for possible future upgrades,
- and finally check the **test cell** at the end for a concrete NSF URL run.

## Practical conclusion upfront
For the **screening-task style requirement** of ingesting a single NSF URL and producing explainable structured output, the strongest default is:

**Deterministic extraction + deterministic tags**, with semantic methods treated as optional augmentation rather than the source of truth.

That keeps the pipeline reproducible, easier to debug, and easier for maintainers to review.


## Comparative view of all approaches

The table below differentiates the approaches specifically in the context of **processing a single NSF opportunity URL**.

| Approach | Core idea | Input used from NSF URL | Determinism | Explainability | Handles wording variation | Dependency / compute cost | Best use in this task | Main limitation |
|---|---|---|---|---|---|---|---|---|
| **1A. Deterministic parser + rule-based ontology tags** | Parse page fields and assign tags from curated ontology rules | Title, description, eligibility, agency | **High** | **High** | Low to Medium | Low | Best default for screening task and maintainers' review | Can miss concepts expressed in unfamiliar wording |
| **1B. Semantic label suggestions** | Compare FOA text against ontology label texts using sentence-transformer embeddings | Same structured FOA text, converted into semantic query text | Medium | Medium | **High** | Medium | Good exploratory layer to discover tags not caught by strict rules | Less transparent than exact rules; threshold tuning matters |
| **1C. Deterministic + semantic fill-in hybrid** | Keep deterministic tags as base, then fill only missing categories from semantic matches | Deterministic output + semantic suggestions | Medium to High | Medium to High | High | Medium | Strong compromise when recall is more important than purity | Hybrid logic can still add borderline tags if thresholds are loose |
| **2. TF-IDF ontology matching** | Represent ontology keywords and FOA text with TF-IDF vectors, then match by cosine similarity | Raw/parsed text from NSF page | Medium | High | Medium | Low to Medium | Useful lightweight baseline beyond plain keyword rules | Still lexical; weak on deeper semantic paraphrases |
| **3A. Embedding similarity tagging** | Encode FOA text and tag descriptions in vector space, assign tags by semantic similarity | Combined title + description + eligibility + optional raw text | Medium | Medium | **High** | Medium to High | Good for semantic generalization and future scaling | More model-dependent and harder to justify tag-by-tag |
| **3B. TF-IDF + embedding hybrid record builder** | Build full FOA record, then apply semantic or statistical tag layer to a richer text corpus | Parsed fields + capped raw page text | Medium | Medium | High | Medium to High | Best experimental path when trying to maximize coverage from one URL | More moving parts; less minimal than the task asks for |

## Recommendation summary

| Scenario | Best choice |
|---|---|
| You want the **most maintainable and reviewable** solution | **Approach 1A** |
| You want a **lightweight non-LLM baseline** beyond pure rules | **Approach 2** |
| You want **better recall on varied NSF wording** | **Approach 3A** or **1C** |
| You want to stay closest to the **original task constraints** | **Approach 1A**, optionally mentioning **1C** as future work |


## Approach 1 — Deterministic extraction with optional semantic augmentation

### Why this approach exists
This is the most task-aligned approach in the notebook.

It starts from a simple principle:
given **one NSF opportunity URL**, extract a clean structured record first, then tag it using a **deterministic ontology-driven pipeline**.

### What this section demonstrates
- a `NSFFOARecord` dataclass for consistent field structure,
- HTML fetching and NSF page parsing,
- deterministic tagging through `RuleBasedTagger`,
- and optional semantic suggestions for exploration.

### Why maintainers may prefer this baseline
- It is **easy to inspect**.
- It is **fully reproducible**.
- It is the easiest approach to align with a requirement like:
  `python main.py --url "<nsf-url>"` and export JSON/CSV.
- Failures are easier to debug because the logic is explicit.

### Expected behavior on NSF URLs
This approach works best when the NSF page exposes enough clear textual signal in:
- title,
- description,
- eligibility,
- agency text.

If the wording is unusual, the deterministic tags may be conservative — which is often acceptable for a screening task because false positives are reduced.


In [56]:
import re
from dataclasses import dataclass, asdict
from typing import Optional, Dict, Any

import requests
from bs4 import BeautifulSoup 

from main import RuleBasedTagger 
import requests
from bs4 import BeautifulSoup
from urllib.parse import urlparse
import re
from typing import Optional, List, Dict
from dateutil import parser as dateparser
import warnings
from datetime import datetime, timezone
import hashlib
from urllib.parse import urljoin
from sentence_transformers import SentenceTransformer, util

In [57]:
!pip -q install sentence-transformers
!pip install lxml


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
import os

# Optional: read HF_TOKEN from the environment for authenticated
# Hugging Face model downloads. Do NOT hard-code tokens in the notebook.
HF_TOKEN = os.getenv("HF_TOKEN")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
else:
    print("HF_TOKEN not set; using unauthenticated access to Hugging Face Hub.")

In [11]:
@dataclass
class NSFFOARecord:
    foa_id: Optional[str] = None
    title: Optional[str] = None
    agency: str = "NSF"
    posting_date: Optional[str] = None
    close_date: Optional[str] = None
    description: Optional[str] = None
    eligibility: Optional[str] = None

    source: str = "nsf.gov"
    source_url: str = ""
    tags: Optional[Dict[str, Any]] = None

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)

In [12]:
def fetch_html(url: str, timeout: int = 30) -> str:
    headers = {"User-Agent": "HumanAI-ISSR4-NSF-Scraper/1.0"}
    resp = requests.get(url, headers=headers, timeout=timeout)
    resp.raise_for_status()
    return resp.text


def parse_nsf_page(url: str) -> NSFFOARecord:
    html = fetch_html(url)
    soup = BeautifulSoup(html, "html.parser")

    # Title (works well for the K-12 AI DCL page)
    title_el = soup.find("h1")
    title = title_el.get_text(strip=True) if title_el else None

    # Short teaser / description: first paragraph under the title if available
    desc_el = soup.find("p")
    description = desc_el.get_text(" ", strip=True) if desc_el else None

    # Fallback to full-page text for regex-based extraction
    full_text = soup.get_text(" ", strip=True)

    # DEADLINES: pull the sentence after the DEADLINES label, if present
    close_date = None
    m_deadline = re.search(r"DEADLINES:\s*(.*?)(PROPOSAL PREPARATION:|COGNIZANT PROGRAM OFFICERS:|$)", full_text, re.S)
    if m_deadline:
        close_date = m_deadline.group(1).strip()

    # ELIGIBILITY: capture the block between ELIGIBILITY and DEADLINES, if present
    eligibility = None
    m_elig = re.search(r"ELIGIBILITY:\s*(.*?)(DEADLINES:|PROPOSAL PREPARATION:|$)", full_text, re.S)
    if m_elig:
        eligibility = m_elig.group(1).strip()

    # Some Dear Colleague Letters (like the BIO-AI one) simply do not
    # include explicit ELIGIBILITY/DEADLINES blocks. In those cases we
    # leave close_date/eligibility as None because the information is
    # not exposed in a structured way on the page.

    # Many NSF Dear Colleague Letters do not expose an obvious posting date
    posting_date = None

    return NSFFOARecord(
        foa_id=None,
        title=title,
        posting_date=posting_date,
        close_date=close_date,
        description=description,
        eligibility=eligibility,
        source_url=url,
    )

In [13]:
# Deterministic NSF FOA tagger wrapper using RuleBasedTagger from main.py

# Reuse the ontology-driven RuleBasedTagger for NSF FOAs.
# This keeps tagging fully deterministic and aligned with the main pipeline.

tagger = RuleBasedTagger()


def tag_nsf_foa(foa: NSFFOARecord) -> Dict[str, Any]:
    """Tag an NSF FOA record using the rule-based ontology.

    Returns a dict with the FOA fields plus a "tags" key so it matches
    the structure used in the Grants.gov pipeline.
    """
    # Combine key text fields to give the ontology enough signal.
    text_parts = [
        foa.title or "",
        foa.description or "",
        foa.eligibility or "",
        foa.agency or "",
    ]
    text = " ".join(t for t in text_parts if t.strip())

    tags = tagger.tag(text)

    rec = foa.to_dict()
    rec["tags"] = tags
    return rec

### Optional semantic extension inside Approach 1

The next cells add a **semantic suggestion layer** using sentence-transformer embeddings over ontology labels.

This is useful for maintainers because it answers an important question:

> If deterministic rules are too strict, can we recover missing categories without replacing the baseline entirely?

The design choice here is intentionally cautious:
- deterministic tags remain the base,
- semantic matches are used only as **suggestions** or **fill-ins**,
- and the notebook keeps the two outputs separate so they can be compared clearly.

This makes the notebook more review-friendly than jumping straight to a black-box model.


In [14]:
# --- Optional helper: semantic tag suggestions via Sentence Transformers ---
# NOTE: This is for notebook exploration only. Your production tags should
# remain deterministic + rule-based via RuleBasedTagger + ontologies.yaml.

# If you don't have it installed in this environment, uncomment:
# !pip -q install sentence-transformers

import yaml
from sentence_transformers import SentenceTransformer, util


def load_ontology_labels(ontology_path: str = "ontologies.yaml"):
    """Load ontology tags and build embedding-ready label texts."""
    with open(ontology_path, "r", encoding="utf-8") as f:
        ont = yaml.safe_load(f) or {}

    categories = ont.get("categories") or {}

    label_texts: list[str] = []
    label_ids: list[tuple[str, str]] = []

    for cat, cfg in categories.items():
        for t in (cfg.get("tags") or []):
            name = str(t.get("name") or "").strip()
            if not name:
                continue
            # Slightly richer label text improves semantic matching.
            label_texts.append(f"{cat} :: {name}")
            label_ids.append((cat, name))

    return label_texts, label_ids


def build_semantic_index(model_name: str = "all-MiniLM-L6-v2", ontology_path: str = "ontologies.yaml"):
    """Create a semantic index over ontology label texts."""
    model = SentenceTransformer(model_name)
    label_texts, label_ids = load_ontology_labels(ontology_path)
    label_embs = model.encode(label_texts, normalize_embeddings=True)
    return model, label_texts, label_ids, label_embs


def suggest_semantic_tags(
    foa: NSFFOARecord,
    model: SentenceTransformer,
    label_ids: list[tuple[str, str]],
    label_embs,
    top_k: int = 5,
) -> list[Dict[str, Any]]:
    """Suggest tags by cosine similarity between FOA text and ontology labels."""
    text_parts = [foa.title or "", foa.description or "", foa.eligibility or "", foa.agency or ""]
    text = " ".join([t for t in text_parts if t.strip()])
    if not text.strip():
        return []

    foa_emb = model.encode(text, normalize_embeddings=True)
    scores = util.cos_sim(foa_emb, label_embs)[0]

    # Get top-k indices
    top = scores.topk(k=min(top_k, len(label_ids)))
    top_idx = top.indices.tolist()

    out: list[Dict[str, Any]] = []
    for i in top_idx:
        cat, name = label_ids[i]
        out.append({
            "category": cat,
            "name": name,
            "semantic_score": float(scores[i]),
        })
    return out


# Build the semantic index (downloads the model the first time)
semantic_model, _label_texts, _label_ids, _label_embs = build_semantic_index()


# Example: compare deterministic tags vs semantic suggestions on the same FOAs
nsf_urls =  "https://www.nsf.gov/funding/opportunities/nsf-ttp-national-science-foundation-translation-practice"



foa = parse_nsf_page(nsf_urls)
det = tag_nsf_foa(foa)  # deterministic, rule-based (official)
sem = suggest_semantic_tags(foa, semantic_model, _label_ids, _label_embs, top_k=5)

results= [{
    "url": nsf_urls,
    "title": foa.title,
    "deterministic_tags": det.get("tags"),
    "semantic_suggestions": sem,
}]

results

f:\proj-HumanAI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4025.54it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[{'url': 'https://www.nsf.gov/funding/opportunities/nsf-ttp-national-science-foundation-translation-practice',
  'title': 'National Science Foundation Translation to Practice (NSF TTP)',
  'deterministic_tags': {'sponsor_org': [{'name': 'NSF'}]},
  'semantic_suggestions': [{'category': 'sponsor_org',
    'name': 'NSF',
    'semantic_score': 0.45986407995224},
   {'category': 'sponsor_themes',
    'name': 'Public Health',
    'semantic_score': 0.304485946893692},
   {'category': 'tech_focus',
    'name': 'Standardization / Interoperability',
    'semantic_score': 0.2955741286277771},
   {'category': 'research_domains',
    'name': 'Computational',
    'semantic_score': 0.2826247811317444},
   {'category': 'sponsor_org',
    'name': 'NIH',
    'semantic_score': 0.2810077369213104}]}]

In [15]:
# --- Hybrid tagging: fill missing categories using best semantic suggestions ---

from copy import deepcopy


def augment_tags_with_semantic(
    deterministic_rec: Dict[str, Any],
    semantic_suggestions: list[Dict[str, Any]],
    max_new_categories: int = 2,
) -> Dict[str, Any]:
    """Fill in missing tag categories using highest-score semantic suggestions.

    - deterministic_rec: output from tag_nsf_foa (includes "tags" from RuleBasedTagger)
    - semantic_suggestions: list of {category, name, semantic_score}
    - We only add categories that are *not* already present in deterministic tags.
    - Scores are used for selection only; we do not serialize them.
    """
    rec = deepcopy(deterministic_rec)
    tags = rec.get("tags") or {}

    existing_cats = set(tags.keys())

    # Sort suggestions by descending semantic_score
    sorted_suggestions = sorted(
        semantic_suggestions,
        key=lambda s: s.get("semantic_score", 0.0),
        reverse=True,
    )

    added = 0
    for s in sorted_suggestions:
        cat = s["category"]
        name = s["name"]
        if cat in existing_cats:
            continue
        tags.setdefault(cat, []).append({"name": name})
        existing_cats.add(cat)
        added += 1
        if added >= max_new_categories:
            break

    rec["tags"] = tags
    return rec


# Example: build hybrid tags for the batch of NSF URLs
hybrid_results = []
for item in results:
    det_tags = item["deterministic_tags"]
    sem_suggestions = item["semantic_suggestions"]

    # Reconstruct a minimal deterministic record so the helper can work
    det_rec = {
        "foa_id": None,
        "title": item["title"],
        "agency": "NSF",
        "posting_date": None,
        "close_date": None,
        "description": None,
        "eligibility": None,
        "source": "nsf.gov",
        "source_url": item["url"],
        "tags": det_tags,
    }

    hybrid = augment_tags_with_semantic(det_rec, sem_suggestions)
    hybrid_results.append({
        "url": item["url"],
        "title": item["title"],
        "hybrid_tags": hybrid["tags"],
    })

hybrid_results

[{'url': 'https://www.nsf.gov/funding/opportunities/nsf-ttp-national-science-foundation-translation-practice',
  'title': 'National Science Foundation Translation to Practice (NSF TTP)',
  'hybrid_tags': {'sponsor_org': [{'name': 'NSF'}],
   'sponsor_themes': [{'name': 'Public Health'}],
   'tech_focus': [{'name': 'Standardization / Interoperability'}]}}]

## Approach 2 — TF-IDF vectorization baseline

### Why include this approach
This section introduces a middle ground between:
- pure rule matching, and
- full semantic embeddings.

TF-IDF is useful as a **classical NLP baseline** because it remains relatively interpretable while allowing softer matching than exact keyword rules.

### What this section demonstrates
- flattening the ontology into text documents,
- vectorizing ontology labels and FOA text with TF-IDF,
- cosine-similarity based tag assignment,
- and running the pipeline on a single NSF URL.

### Why it matters for maintainers
This approach shows that the notebook is not limited to one hand-written rule system.  
It evaluates whether a low-cost statistical method can improve coverage on NSF pages that use slightly different terminology.

### Strength of this baseline
TF-IDF is attractive when you want:
- low additional complexity,
- no large hosted model dependency,
- and reasonably interpretable matching behavior.

### Limitation to keep in mind
Even though it is stronger than exact rules for lexical overlap, it still depends heavily on surface wording and usually cannot capture semantic equivalence as well as embeddings.


In [16]:
ONTOLOGY = {
    # ── AXIS 1: What scientific field? ──
    "research_domains": {
        "health_biomedical": [
            "health", "biomedical", "clinical", "disease", "cancer",
            "mental health", "public health", "nursing", "medicine",
            "epidemiology", "genomics", "neuroscience", "therapeutics",
            "pharmacology", "medical", "patient", "diagnosis"
        ],
        "social_sciences": [
            "social science", "sociology", "psychology", "anthropology",
            "political science", "economics", "behavioral", "cognitive",
            "demography", "criminology"
        ],
        "education": [
            "education", "learning", "curriculum", "pedagogy",
            "k-12", "higher education", "stem education", "literacy",
            "classroom", "teacher", "student achievement"
        ],
        "environment": [
            "environment", "climate", "ecology", "sustainability",
            "biodiversity", "water", "energy", "carbon", "pollution",
            "conservation", "renewable", "greenhouse"
        ],
        "engineering_technology": [
            "engineering", "robotics", "manufacturing", "materials",
            "infrastructure", "transportation", "aerospace", "semiconductor"
        ],
        "computer_science": [
            "artificial intelligence", "machine learning", "data science",
            "cybersecurity", "software", "algorithm", "computing",
            "computer science", "deep learning", "neural network",
            "natural language processing", "computer vision"
        ],
        "arts_humanities": [
            "humanities", "history", "linguistics", "literature",
            "philosophy", "cultural", "arts", "music", "archaeology"
        ],
    },

    # ── AXIS 2: How is research done? ──
    "methods": {
        "survey": [
            "survey", "questionnaire", "interview", "focus group"
        ],
        "randomized_controlled": [
            "randomized", "rct", "controlled trial", "experimental design"
        ],
        "computational": [
            "computational", "simulation", "modeling", "algorithm",
            "machine learning", "deep learning", "neural network", "data-driven"
        ],
        "qualitative": [
            "qualitative", "ethnograph", "case study", "grounded theory", "thematic"
        ],
        "longitudinal": [
            "longitudinal", "cohort", "follow-up", "prospective", "panel study"
        ],
        "meta_analysis": [
            "meta-analysis", "systematic review", "meta analysis"
        ],
        "mixed_methods": [
            "mixed methods", "mixed-methods"
        ],
        "observational": [
            "observational", "cross-sectional", "retrospective"
        ],
    },

    # ── AXIS 3: Who is studied/served? ──
    "populations": {
        "underserved_communities": [
            "underserved", "disadvantaged", "low-income", "poverty",
            "under-resourced", "marginalized", "underrepresented"
        ],
        "rural": [
            "rural", "remote", "frontier", "non-urban"
        ],
        "minority": [
            "minority", "racial", "ethnic", "indigenous", "tribal",
            "hispanic", "african american", "black", "asian american", "native"
        ],
        "youth_children": [
            "youth", "children", "adolescent", "pediatric", "k-12",
            "juvenile", "infant", "toddler"
        ],
        "elderly": [
            "elderly", "older adult", "aging", "geriatric", "senior"
        ],
        "veterans": [
            "veteran", "military", "armed forces", "service member"
        ],
        "women": [
            "women", "gender", "female", "maternal", "pregnancy"
        ],
        "persons_with_disabilities": [
            "disability", "disabilities", "disabled", "impairment", "accessibility"
        ],
    },

    # ── AXIS 4: What is the funder's goal? ──
    "sponsor_themes": {
        "workforce_development": [
            "workforce", "job training", "employment", "career",
            "apprenticeship", "upskilling", "vocational"
        ],
        "equity_inclusion": [
            "equity", "inclusion", "diversity", "belonging",
            "dei", "equal opportunity", "broadening participation"
        ],
        "innovation": [
            "innovation", "entrepreneurship", "commercialization",
            "startup", "technology transfer", "patent"
        ],
        "capacity_building": [
            "capacity building", "institutional capacity", "capability development"
        ],
        "community_engagement": [
            "community engagement", "community-based", "participatory",
            "outreach", "partnership", "stakeholder"
        ],
        "national_security": [
            "national security", "defense", "homeland", "intelligence"
        ],
    }
}

In [17]:
docs = []
labels = []

for axis, categories in ONTOLOGY.items():
    for label, keywords in categories.items():
        docs.append(" ".join(keywords))
        labels.append((axis, label))

In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer()

text = tfidf.fit_transform(docs)

In [19]:
text.shape

(29, 217)

In [38]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def flatten_ontology(ontology):
    docs = []
    meta = []

    for axis, categories in ontology.items():
        for label, keywords in categories.items():
            doc = f"{label.replace('_',' ')} " + " ".join(keywords)
            docs.append(doc)
            meta.append({
                "axis": axis,
                "label": label
            })
    return docs, meta


def build_foa_text(foa):

    parts = [
        foa.get("title", ""),
        foa.get("description", ""),
        foa.get("eligibility", ""),
        foa.get("agency", "")
    ]

    return " ".join(p for p in parts if p).strip()


def tfidf_tagging(foa, ontology, threshold=0.15):
    foa_text = build_foa_text(foa)
    ontology_docs, ontology_meta = flatten_ontology(ontology)
    tfidf = TfidfVectorizer()
    matrix = tfidf.fit_transform(ontology_docs)
    foa_vector = tfidf.transform([foa_text])
    scores = cosine_similarity(foa_vector, matrix)[0]
    tags = {axis: [] for axis in ontology}
    for meta, score in zip(ontology_meta, scores):
        if score >= threshold:
            axis = meta["axis"]
            label = meta["label"]
            tags[axis].append((label, score))

    # sort tags by score
    for axis in tags:
        tags[axis] = sorted(tags[axis], key=lambda x: -x[1])
        tags[axis] = [t[0] for t in tags[axis]]

    return tags

In [46]:
DATE_RE = re.compile(
    r"\b("
    r"\d{1,2}[/-]\d{1,2}[/-]\d{2,4}|"
    r"\d{1,2}[-\s](?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*[-\s]\d{2,4}|"
    r"(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*\.?\s+\d{1,2}(?:st|nd|rd|th)?,?\s+\d{4}|"
    r"\d{4}-\d{2}-\d{2}"
    r")\b",
    re.I,
)


def normalize_date(raw: Optional[str]) -> Optional[str]:
    if not raw:
        return None
    value = str(raw).strip()
    if not value:
        return None
    if dateparser is not None:
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                return dateparser.parse(value).strftime("%Y-%m-%d")
        except Exception:
            pass
    for fmt in ("%Y-%m-%d", "%m/%d/%Y", "%m/%d/%y", "%m-%d-%Y", "%m-%d-%y",
                "%b %d, %Y", "%B %d, %Y", "%d-%b-%Y", "%d-%B-%Y"):
        try:
            return datetime.strptime(value, fmt).strftime("%Y-%m-%d")
        except ValueError:
            continue
    return None

In [47]:


def extract_nsf_title(soup):
    h1 = soup.find("h1")
    if h1:
        return h1.get_text(strip=True)
    meta = soup.select_one('meta[property="og:title"]')
    if meta:
        return meta["content"]
    return ""

def extract_nsf_deadlines(lines):
    open_date = None
    close_date = None
    for i, line in enumerate(lines):
        l = line.lower()
        if "posted on" in l and i + 1 < len(lines):
            open_date = normalize_date(lines[i + 1])
        if "full proposal deadline" in l and i + 1 < len(lines):
            close_date = normalize_date(lines[i + 1])
    return open_date, close_date

def extract_nsf_award(text):
    if not text:
        return None
    m = re.search(
        r"between\s*\$([\d,]+)[^\$]*and\s*\$([\d,]+)",
        text,
        re.I
    )
    if m:
        return {
            "min": int(m.group(1).replace(",", "")),
            "max": int(m.group(2).replace(",", ""))
        }
    m = re.search(
        r"\$([\d,]+)\s*(?:to|-)\s*\$([\d,]+)",
        text,
        re.I
    )
    if m:
        return {
            "min": int(m.group(1).replace(",", "")),
            "max": int(m.group(2).replace(",", ""))
        }
    m = re.search(
        r"up to\s*\$([\d,]+)",
        text,
        re.I
    )
    if m:
        return {
            "min": None,
            "max": int(m.group(1).replace(",", ""))
        }
    m = re.search(
        r"maximum \s*\$([\d,]+)",
        text,
        re.I
    )
    if m:
        return {
            "min": None,
            "max": int(m.group(1).replace(",", ""))
        }

    return None

def extract_nsf_id(text):
    m = re.search(r"\b(NSF\s*\d{2}-\d{3})\b", text)
    if m:
        return m.group(1)
    m = re.search(r"\b(DCL\s*\d{2}-\d{3})\b", text)
    if m:
        return m.group(1)
    return ""

def extract_nsf_section(soup, keywords):
    for tag in soup.find_all(["h2", "h3", "strong"]):
        txt = tag.get_text(" ", strip=True).lower()
        if any(k.lower() in txt for k in keywords):
            parts = []
            for sib in tag.parent.find_next_siblings():
                if sib.name in ["h2", "h3"]:
                    break
                t = sib.get_text(" ", strip=True)
                if t:
                    parts.append(t)
            if parts:
                return " ".join(parts)[:3000]
    return ""

def extract_nsf_deadlines_from_text(text):

    open_date = None
    close_date = None

    for match in re.finditer(
        r"(posted|published).*?(" + DATE_RE.pattern + ")",
        text,
        re.I | re.S
    ):
        open_date = normalize_date(match.group(2))

    for match in re.finditer(
        r"(proposal deadline|full proposal deadline).*?(" + DATE_RE.pattern + ")",
        text,
        re.I | re.S
    ):
        close_date = normalize_date(match.group(2))

    return open_date, close_date

def extract_nsf_eligibility(soup):
    for tag in soup.find_all(["h2","h3"]):
        txt = tag.get_text(" ",strip=True).lower()
        if "eligibility information" in txt or "who may submit proposals" in txt:
            parts = []
            for sib in tag.find_next_siblings():
                if sib.name in ["h2","h3"]:
                    break
                t = sib.get_text(" ",strip=True)
                if t:
                    parts.append(t)
            if parts:
                return " ".join(parts)[:2000]

    return ""
def find_nsf_solicitation_url(soup, base_url):

    for a in soup.find_all("a", href=True):

        href = a["href"]

        if "solicitation" in href.lower():
            return urljoin(base_url, href)

        if "nsf" in href.lower() and "solicitation" in a.get_text().lower():
            return urljoin(base_url, href)

    return None

In [48]:
from urllib.parse import urlparse

DOMAIN_AGENCY_MAP = {
    "grants.gov":"Grants.gov",
    "nsf.gov":"National Science Foundation",
}

def get_agency_name(url):
    hostname = urlparse(url).hostname
    if not hostname:
        return ""
    hostname = hostname.lower()
    for domain, agency in DOMAIN_AGENCY_MAP.items():
        if hostname == domain or hostname.endswith(f".{domain}"):
            return agency
    return hostname


In [49]:
def get_html_soup(html: str):
    try:
        return BeautifulSoup(html, "lxml")
    except Exception:
        return BeautifulSoup(html, "html.parser")

In [50]:
_BROWSER_UA = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/120.0.0.0 Safari/537.36"
)


def http_get(url: str, timeout: int = 20, accept_json: bool = False):
    hdrs = {"User-Agent": _BROWSER_UA}
    if accept_json:
        hdrs["Accept"] = "application/json"
    resp = requests.get(url, headers=hdrs, timeout=timeout)
    resp.raise_for_status()
    return resp

In [52]:
def scrape_nsf(url, timeout=20):

    resp = http_get(url, timeout)
    soup = get_html_soup(resp.text)

    text = soup.get_text("\n", strip=True)
    lines = [l.strip() for l in text.split("\n") if l.strip()]

    title = extract_nsf_title(soup)
    foa_id = extract_nsf_id(text)

    text = soup.get_text(" ", strip=True)

    open_date, close_date = extract_nsf_deadlines_from_text(text)

    if not open_date or not close_date:

        sol_url = find_nsf_solicitation_url(soup, url)

        if sol_url:
            resp2 = http_get(sol_url, timeout)
            soup2 = get_html_soup(resp2.text)

            text2 = soup2.get_text(" ", strip=True)

            open_date2, close_date2 = extract_nsf_deadlines_from_text(text2)

            open_date = open_date or open_date2
            close_date = close_date or close_date2

    description = extract_nsf_section(soup, ["Synopsis"])

    eligibility = extract_nsf_section(
        soup,
        ["Eligibility Information", "Who May Submit Proposals"]
    )

    agency_name = get_agency_name(url)

    # If eligibility missing → check solicitation page
    if not eligibility:

        sol_url = find_nsf_solicitation_url(soup, url)

        if sol_url:
            resp2 = http_get(sol_url, timeout)
            soup2 = get_html_soup(resp2.text)

            eligibility = extract_nsf_section(
                soup2,
                ["Eligibility Information", "Who May Submit Proposals"]
            )

    award_range = extract_nsf_award(text)

    if not award_range:

        sol_url = find_nsf_solicitation_url(soup, url)

        if sol_url:
            resp2 = http_get(sol_url, timeout)
            soup2 = get_html_soup(resp2.text)

            text2 = soup2.get_text(" ", strip=True)

            award_range = extract_nsf_award(text2)

    raw = " ".join(filter(None, [title, description, eligibility]))
    result = {
        "foa_id": foa_id,
        "title": title,
        "agency": agency_name,
        "open_date": open_date,
        "close_date": close_date,
        "eligibility": eligibility,
        "description": description,
        "award_range": award_range,
        "source_url": url,
        "_raw_text": raw
    }


    return result

In [53]:
def run_tfidf_pipeline(url):
    foa = scrape_nsf(url)
    tags = tfidf_tagging(foa, ONTOLOGY)

    print("──────────────────────────────────────────────────")
    print("FOA SUMMARY")
    print("──────────────────────────────────────────────────")

    print("ID:      ", foa["foa_id"])
    print("Title:   ", foa["title"][:70])
    print("Agency:  ", foa["agency"])
    print("Opens:   ", foa["open_date"])
    print("Closes:  ", foa["close_date"])
    print("Award:   ", foa["award_range"])

    print("\nTAGS:")
    for axis, labels in tags.items():
        if labels:
            print(f"  {axis}: {labels}")

In [54]:
import requests
import json
from bs4 import BeautifulSoup
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


# ─────────────────────────────────────────────
# ONTOLOGY FLATTENER
# ─────────────────────────────────────────────

def flatten_ontology(ontology):

    docs = []
    meta = []

    for axis, categories in ontology.items():
        for label, keywords in categories.items():

            doc = f"{label.replace('_',' ')} " + " ".join(keywords)

            docs.append(doc)

            meta.append({
                "axis": axis,
                "label": label
            })

    return docs, meta


# Build ontology vectors ONCE
ONTOLOGY_DOCS, ONTOLOGY_META = flatten_ontology(ONTOLOGY)

TFIDF = TfidfVectorizer()
ONTOLOGY_MATRIX = TFIDF.fit_transform(ONTOLOGY_DOCS)


# ─────────────────────────────────────────────
# FOA TEXT BUILDER
# ─────────────────────────────────────────────

def build_foa_text(foa):

    parts = [
        foa.get("title", ""),
        foa.get("description", ""),
        foa.get("eligibility", ""),
        foa.get("agency", "")
    ]

    return " ".join(p for p in parts if p).strip()


# ─────────────────────────────────────────────
# TFIDF TAGGER
# ─────────────────────────────────────────────

def tag_foa_with_tfidf(foa_text, threshold=0.15):

    foa_vec = TFIDF.transform([foa_text])

    scores = cosine_similarity(foa_vec, ONTOLOGY_MATRIX)[0]

    tags = {axis: [] for axis in ONTOLOGY}

    for meta, score in zip(ONTOLOGY_META, scores):

        if score >= threshold:

            axis = meta["axis"]
            label = meta["label"]

            tags[axis].append((label, score))

    # sort tags by score
    for axis in tags:

        tags[axis] = sorted(tags[axis], key=lambda x: -x[1])

        tags[axis] = [{"name": t[0]} for t in tags[axis]]

    return tags


# ─────────────────────────────────────────────
# OUTPUT BUILDER
# ─────────────────────────────────────────────

def build_final_output(foa, tags):

    return {
        "title": foa.get("title", ""),
        "agency": foa.get("agency", ""),
        "description": foa.get("description", ""),
        "eligibility": foa.get("eligibility", ""),
        "source_url": foa.get("source_url", ""),
        "tags": tags
    }


# ─────────────────────────────────────────────
# PIPELINE
# ─────────────────────────────────────────────

def run_pipeline(url):

    foa = scrape_nsf(url)

    foa_text = build_foa_text(foa)

    tags = tag_foa_with_tfidf(foa_text)

    result = build_final_output(foa, tags)

    print(json.dumps(result, indent=2, ensure_ascii=False))

    return result


# ─────────────────────────────────────────────
# RUN
# ─────────────────────────────────────────────

if __name__ == "__main__":
    nsf_url = "https://www.nsf.gov/funding/opportunities/nsf-ttp-national-science-foundation-translation-practice"
    run_pipeline(nsf_url)

{
  "title": "National Science Foundation Translation to Practice (NSF TTP)",
  "agency": "National Science Foundation",
  "description": "The U.S. NSF Directorate for Technology, Innovation and Partnerships (NSF TIP) partners across sectors to advance three primary focus areas – accelerating technology translation and development, fostering regional innovation and economic growth, and preparing the American workforce for future high-wage jobs in STEM fields. The translation of research to practice ensures that the insights and innovations developed through scientific study and experimentation have tangible, positive impacts for the Nation. These impacts include improving the quality of life, promoting economic and job growth, ensuring national security, and maintaining global competitiveness. Indeed, scientific and engineering breakthroughs have the potential to address critical societal challenges in industries such as aerospace, agriculture, communications, education, energy, health

## Approach 3 — Embedding-based tagging and hybridization

### Why this approach exists
NSF opportunities often vary in wording, even when they point to similar scientific domains or funding themes.

This section explores whether **semantic embeddings** can improve tag recovery when:
- the ontology wording and the page wording do not overlap exactly,
- or the concept is present but expressed indirectly.

### What this section demonstrates
- building tag-description embeddings,
- encoding FOA text into the same vector space,
- assigning tags by similarity score,
- and combining statistical / semantic signals into a richer FOA record.

### Why this is valuable to maintainers
This is the strongest section for showing **forward-looking extensibility**.  
It demonstrates that the current deterministic notebook can evolve into a more semantically aware pipeline without rewriting the entire ingestion flow.

### Key trade-off
This approach typically improves recall, but it also introduces:
- threshold sensitivity,
- more dependencies,
- and slightly weaker explainability compared with a fully deterministic baseline.

For the actual task, this makes it a strong **research/next-step direction**, even if not the safest default for final production output.


In [55]:
TAG_DESCRIPTIONS = {}
TAG_TO_CATEGORY = {}

for category, tags in ONTOLOGY.items():
    for tag, keywords in tags.items():
        TAG_DESCRIPTIONS[tag] = " ".join(keywords)
        TAG_TO_CATEGORY[tag] = category

In [58]:
import torch
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7683.38it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [65]:
tag_names = list(TAG_DESCRIPTIONS.keys())

tag_matrix = torch.stack([
    model.encode(TAG_DESCRIPTIONS[tag], convert_to_tensor=True)
    for tag in tag_names
])

In [66]:
def build_embedding_text(foa):

    if isinstance(foa, str):
        return foa

    if "_raw_text" in foa:
        return foa["_raw_text"]

    return " ".join(filter(None, [
        foa.get("title",""),
        foa.get("description",""),
        foa.get("eligibility","")
    ]))

In [67]:
def apply_embedding_tags(text, threshold=0.42):

    text = text[:1000]

    text_emb = model.encode(text, convert_to_tensor=True)

    sims = util.cos_sim(text_emb, tag_matrix)[0]

    results = []

    for i, score in enumerate(sims):

        if score >= threshold:
            results.append((tag_names[i], float(score)))

    return sorted(results, key=lambda x: -x[1])

In [68]:
from typing import Dict, List


def apply_tags(text: str, threshold: float = 0.15) -> Dict[str, List[str]]:
    """Wrapper that applies the TFIDF-based ontology tagger to free text.

    Returns a simple mapping of axis -> [tag_name, ...] so it can be
    combined with embedding-based tags in `hybrid_tags`.
    """
    tags_dict = tag_foa_with_tfidf(text, threshold=threshold)
    return {
        axis: [t["name"] for t in labels] if labels else []
        for axis, labels in tags_dict.items()
    }

In [69]:
def hybrid_tags(foa):

    text = build_embedding_text(foa)

    rule_tags = apply_tags(text)

    embed_tags = apply_embedding_tags(text)

    merged = {cat: set(tags) for cat, tags in rule_tags.items()}

    for tag, score in embed_tags:

        category = TAG_TO_CATEGORY[tag]

        if category not in merged:
            merged[category] = set()

        merged[category].add(tag)

    return {k: sorted(list(v)) for k,v in merged.items()}

In [ ]:
def hybrid_tags(foa):

    text = build_embedding_text(foa)

    rule_tags = apply_tags(text)

    embed_tags = apply_embedding_tags(text)

    merged = {cat: set(tags) for cat, tags in rule_tags.items()}

    for tag, score in embed_tags:

        category = TAG_TO_CATEGORY[tag]

        if category not in merged:
            merged[category] = set()

        merged[category].add(tag)

    return {k: sorted(list(v)) for k,v in merged.items()}

In [74]:
def build_foa_record(url: str) -> dict:
    """
    Full pipeline:
    1. Ingest from web
    2. Build tag corpus from all text fields
    3. Apply semantic tags
    4. Clean up internal fields
    5. Return final record
    """
    # Step 1: Ingest
    raw = scrape_nsf(url)

    # Step 2: Build tag corpus
    # Combine: title + description + eligibility + raw page text
    # We use title + description + eligibility even though they're
    # subsets of raw_text, because they're cleaner/more focused
    raw_text = raw.pop("_raw_text", "")
    tag_corpus = " ".join(filter(None, [
        raw.get("title", ""),
        raw.get("description", ""),
        raw.get("eligibility", ""),
        raw_text[:5000]  # Cap to first 5000 chars of full page
    ]))

    # Step 3: Apply semantic tags
    raw["tags"] = apply_tags(tag_corpus)

    # Step 4: Add metadata
    raw["ingested_at"] = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

    return raw

In [75]:
def test_nsf_url(url: str):
    """Convenience helper to test the full NSF tagging stack on a single URL.

    For the given NSF FOA URL, this will:
    - scrape the page,
    - compute deterministic rule-based tags (via tag_nsf_foa),
    - compute semantic suggestions from the ontology label embeddings,
    - build hybrid tags that fill in missing categories using embeddings,
    - run the TF‑IDF + embedding hybrid (build_foa_record),
    - and print a concise summary.

    Returns a dict with all intermediate results for further inspection.
    """
    # Scrape and build NSFFOARecord for deterministic + semantic comparison
    foa_struct = parse_nsf_page(url)
    det = tag_nsf_foa(foa_struct)
    sem = suggest_semantic_tags(foa_struct, semantic_model, _label_ids, _label_embs, top_k=5)

    # Build hybrid tags using deterministic + semantic suggestions
    det_rec = det.copy()
    hybrid = augment_tags_with_semantic(det_rec, sem)

    # Run the TF‑IDF + embedding hybrid pipeline
    foa_full = build_foa_record(url)

    summary = {
        "url": url,
        "title": foa_struct.title,
        "deterministic_tags": det.get("tags"),
        "semantic_suggestions": sem,
        "hybrid_tags_deterministic_plus_semantic": hybrid.get("tags"),
        "tfidf_embedding_record": foa_full,
    }

    # Pretty-print a short view for quick checks
    print("URL:", url)
    print("Title:", foa_struct.title)
    print("\nDeterministic tags:")
    print(det.get("tags"))

    print("\nTop semantic suggestions:")
    for s in sem:
        print(f"  {s['category']} :: {s['name']} (score={s['semantic_score']:.3f})")

    print("\nHybrid tags (deterministic + semantic fill-in):")
    print(hybrid.get("tags"))

    print("\nTF‑IDF + embedding hybrid tags (from build_foa_record):")
    print(foa_full.get("tags"))

    return summary

In [76]:
result = test_nsf_url(
    "https://www.nsf.gov/funding/opportunities/nsf-ttp-national-science-foundation-translation-practice"
)

URL: https://www.nsf.gov/funding/opportunities/nsf-ttp-national-science-foundation-translation-practice
Title: National Science Foundation Translation to Practice (NSF TTP)

Deterministic tags:
{'sponsor_org': [{'name': 'NSF'}]}

Top semantic suggestions:
  sponsor_org :: NSF (score=0.460)
  sponsor_themes :: Public Health (score=0.304)
  tech_focus :: Standardization / Interoperability (score=0.296)
  research_domains :: Computational (score=0.283)
  sponsor_org :: NIH (score=0.281)

Hybrid tags (deterministic + semantic fill-in):
{'sponsor_org': [{'name': 'NSF'}], 'sponsor_themes': [{'name': 'Public Health'}], 'tech_focus': [{'name': 'Standardization / Interoperability'}]}

TF‑IDF + embedding hybrid tags (from build_foa_record):
{'research_domains': ['education'], 'methods': [], 'populations': [], 'sponsor_themes': ['national_security', 'innovation']}


## Final maintainer note

This notebook is intentionally written as an **evaluation notebook**, not just a final answer dump.

### What I want maintainers to take away
- I started from the **task-aligned deterministic solution**.
- I explored **lighter statistical** and **semantic** alternatives to test robustness on NSF wording.
- I kept the approaches separated so trade-offs remain visible.
- I treated hybridization as an **augmentation strategy**, not as a replacement for an explainable baseline.

### My current recommendation
If this work is converted into the final minimal submission for ISSR4, I would keep:

1. **Deterministic NSF parsing**
2. **Deterministic rule-based tagging as the primary output**
3. **Optional semantic or TF-IDF augmentation only as a documented enhancement path**

That gives the best balance of:
- simplicity,
- reproducibility,
- alignment with the task,
- and maintainability for future contributors.
